In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a model locally:
   ```bash
   ollama run llama3
   ```

   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test the local server, run:
   ```bash
   curl http://localhost:11434
   ```

   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I’m not seeing any FAQ entry about **Olama/Ollama** specifically.

The closest related note is that you **can run the course locally** instead of using Codespaces, as long as you’re comfortable setting up the needed tools such as **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. If you do run locally, you should **document your setup and keep it reproducible**.

If you meant a specific local setup for Ollama, it isn’t covered in the provided context.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Possibly — it depends on the course’s enrollment policy and whether it’s still open.\n\nHere’s what to check quickly:\n- **Course status:** Is it currently open for registration or waitlist?\n- **Deadlines:** Has the add/drop or enrollment deadline passed?\n- **Prerequisites:** Do you meet any required background or eligibility rules?\n- **Capacity:** Is there still space, or can the instructor approve an override?\n- **Format:** Is it self-paced, cohort-based, or scheduled live sessions?\n\nIf you want, send me:\n1. the **course name/link**, and  \n2. the **platform/school** it’s on,  \n\nand I can help you figure out whether you can still join and what to do next.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join enrollment late registration"}', call_id='call_hYH9nuCv4rpCiR3mIHoCVZsW', name='search', type='function_call', id='fc_0d8db191602d0ad2006a375bc62b8c819fb5b1277cd7e1213f', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course late discovered the course join enrollment late registration'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join enrollment late registration"}', call_id='call_hYH9nuCv4rpCiR3mIHoCVZsW', name='search', type='function_call', id='fc_0d8db191602d0ad2006a375bc62b8c819fb5b1277cd7e1213f', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_hYH9nuCv4rpCiR3mIHoCVZsW',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer":

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join and start learning now.

If you want a certificate, though, you’ll need to submit your project while submissions are still open.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(779, 36)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [27]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [28]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment register late join FAQ"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
function_call: search {"query":"can I still join the course after it starts FAQ"}


In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment register late join FAQ"}', call_id='call_AK7L7Ydldw8ggIb8PO7IRqf4', name='search', type='function_call', id='fc_0d53d6dc2530e8d1006a376037230081a199259e51ead5a6b2', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='ca

In [30]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration"}
function_call: search {"query":"new student join course after start enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. If you want a certificate, make sure you submit your project while submissions are still open.

If you’d like, I can also help with the next steps for starting the course or explain the certificate requirements.


In [31]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [35]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can join even if you just discovered the course. If you want a certificate, you’ll need to submit your project while submissions are still open.

You can also start learning and submit homework while the course form/platform is open, even if you didn’t register beforehand. Registration is mainly for gauging interest.

If you want, I can also help with how to get started or explain the certificate requirements.


In [36]:
result

'Yes — you can join even if you just discovered the course. If you want a certificate, you’ll need to submit your project while submissions are still open.\n\nYou can also start learning and submit homework while the course form/platform is open, even if you didn’t register beforehand. Registration is mainly for gauging interest.\n\nIf you want, I can also help with how to get started or explain the certificate requirements.'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen’s gambit.” It looks like that question is off-topic for the course.

If you meant something course-related, feel free to rephrase it, and I’ll look it up. Are there other areas you want to explore?


In [38]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [39]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [40]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [41]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [42]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [43]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [44]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [45]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [46]:
result.cost

CostInfo(input_cost=Decimal('0.0011415'), output_cost=Decimal('0.001188'), total_cost=Decimal('0.0023295'))

In [47]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama locally"}', call_id='call_SggkmQ

In [48]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run();

You: docker in docker run docker in docker dind


-> Response received


-> Response received


-> Response received
